# 🗣️ PLN — Processamento de Linguagem Natural com Machine Learning
## Um Guia Completo e Bem Estruturado

---

### 📚 Sumário

| Parte | Tópico | Descrição |
|-------|--------|-----------|
| **1** | Introdução ao PLN | O que é, exemplos do dia a dia, desafios e pipeline geral |
| **2** | Setup e Instalações | Dependências, imports e downloads do NLTK |
| **3** | Tokenização | Dividir texto em unidades menores (palavras, sentenças) e normalizar |
| **4** | Stopwords | Remoção de palavras irrelevantes — e quando **não** removê-las |
| **5** | Bag of Words (BoW) | Representação numérica de texto por contagem de frequências |
| **6** | TF-IDF | Pesos inteligentes que destacam palavras realmente relevantes |
| **7** | BoW vs TF-IDF | Comparação prática lado a lado com o mesmo corpus |
| **8** | Classificação de Sentimentos | Naive Bayes e Logistic Regression aplicados a reviews |
| **9** | Word Embeddings | Vetores semânticos densos (Word2Vec) — conceito e demo |
| **10** | Conclusão | Resumo do pipeline, checklist e próximos passos |

> **Público-alvo:** Estudantes de ADS com base em Python/SQL, começando em IA/PLN.  
> Cada seção responde três perguntas: **O que é?** → **Por que importa?** → **Como usar na prática?**


---
# PARTE 1: Introdução ao PLN

## 🚀 O que é PLN (Processamento de Linguagem Natural)?

**PLN** é a área da Inteligência Artificial que permite computadores **entenderem, interpretarem e gerarem texto** escrito ou falado por seres humanos. Ele une três campos:

- **Linguística** — como as línguas funcionam (gramática, semântica, pragmática)
- **Ciência da Computação** — algoritmos, estruturas de dados e otimização
- **Inteligência Artificial** — aprendizado de máquina, reconhecimento de padrões

### Exemplos de PLN no seu dia a dia
| Situação | O que o PLN faz por trás |
|----------|------------------------|
| 🔍 Google: "melhor padaria perto de mim" | Interpreta a intenção e localização |
| 📱 WhatsApp sugere "Estou chegando" | Prevê respostas com base no contexto |
| 🎤 Alexa/Siri entendem voz | Converte fala → texto → ação |
| 💬 ChatGPT responde perguntas | Gera texto coerente a partir de contexto |
| 📧 Gmail filtra spam | Classifica e-mails pelo conteúdo |

### Desafios do PLN 🤔

- ❓ **Ambiguidade:** "banco" = instituição financeira ou assento? Depende do contexto!
- 😊 **Ironia/sarcasmo:** "Que produto *maravilhoso*" pode ser negativo.
- 🌐 **Negação:** "não é ruim" → na verdade é positivo, mas contém "ruim".
- 🗨️ **Variações:** "adorei", "amei", "top", "show" transmitem a mesma ideia.

### Pipeline de PLN (Fluxo Geral) 🔀

Este notebook segue **exatamente esse fluxo**, passo a passo:

```
Texto Bruto  (string crua)
    ↓ 🔤 Tokenização  (dividir em palavras)          ← PARTE 3
Tokens
    ↓ 🔡 Normalização  (tudo minúsculo)               ← PARTE 3
Tokens normalizados
    ↓ ✂️ Stopwords  (remover palavras genéricas)       ← PARTE 4
Texto limpo
    ↓ 📊 Vetorização  (converter texto em números)     ← PARTES 5, 6 e 7
Matriz numérica
    ↓ 🤖 Modelo  (classificar, agrupar, predizer)     ← PARTE 8
Predição
```

> 👉 **Conexão com a próxima parte:** Antes de começar a processar texto, precisamos instalar e importar as bibliotecas necessárias. Vamos ao **Setup**!


---
# PARTE 2: Setup e Instalações

## 🛠️ Bibliotecas Principais

| Biblioteca | Para que usamos neste notebook |
|------------|-------------------------------|
| **NLTK** | Tokenização (`word_tokenize`, `sent_tokenize`), lista de stopwords |
| **Scikit-learn** | `CountVectorizer` (BoW), `TfidfVectorizer`, `MultinomialNB`, `LogisticRegression` |
| **Pandas** | Visualizar matrizes BoW/TF-IDF como tabelas legíveis (`DataFrame`) |
| **NumPy** | Operações numéricas com vetores e matrizes |
| **Gensim** | Treinar Word2Vec (Word Embeddings) |

> Execute as duas células abaixo **apenas uma vez** no seu ambiente (Colab/Jupyter).


In [ ]:
# =============================================
# INSTALAÇÃO DE DEPENDÊNCIAS
# =============================================
# Por que instalar?
#   Essas bibliotecas NÃO vêm por padrão no Python.
#   Execute esta célula apenas uma vez; depois pode comentá-la.

!pip install nltk scikit-learn pandas numpy gensim


In [ ]:
# =============================================
# IMPORTAÇÕES E DOWNLOADS NLTK
# =============================================
# Centralizamos todos os imports aqui no início.
# Assim, se faltar alguma biblioteca, o erro aparece logo (fail fast).

import nltk                                   # Toolkit de PLN
import numpy as np                             # Operações numéricas
import pandas as pd                            # DataFrames para visualização
from sklearn.feature_extraction.text import (  # Vetorizadores
    CountVectorizer,
    TfidfVectorizer,
)

print("✅ Importações básicas concluídas!")

# Baixar recursos essenciais do NLTK (executar apenas 1 vez)
# - punkt_tab  → modelo de tokenização (dividir texto em palavras/sentenças)
# - stopwords  → lista de palavras comuns por idioma (pt, en, etc.)
# - wordnet    → base léxica usada para sinônimos e lemas
for recurso in ['punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(recurso, quiet=True)  # quiet=True evita log excessivo

print("✅ Recursos NLTK baixados!")
print("\n👉 Tudo pronto! Próximo passo: Tokenização — o primeiro passo real do pipeline.")


---
# PARTE 3: Tokenização em Profundidade

## ✂️ O que é Tokenização?

**Definição:** Tokenização é o processo de dividir um texto cru em unidades menores chamadas **tokens**.  
Um token pode ser uma palavra, número, pontuação ou até emoji.

### Por que é importante no pipeline?

É o **primeiro passo obrigatório**. Sem tokenização, o computador enxerga o texto como uma única string, sem saber onde uma palavra termina e outra começa:

```
❌ SEM tokenização:
   Computador vê: "O Brasil ganhou! Que jogo."
   → Uma string só. Como contar palavras? Como buscar padrões?

✅ COM tokenização:
   Computador vê: ["O", "Brasil", "ganhou", "!", "Que", "jogo", "."]
   → Cada elemento é analisável individualmente.
```

### Tipos de Tokenização

| Tipo | O que faz | Quando usar |
|------|----------|-------------|
| **Por palavras** | Separa cada palavra e pontuação | Maioria dos casos de PLN |
| **Por sentenças** | Separa frases completas | Resumo automático, análise por frase |
| **Por caracteres** | Separa letra por letra | Idiomas sem espaço (chinês), subpalavras |

### O que vamos fazer na célula abaixo:
1. **Tokenizar por palavras** — com `word_tokenize` (NLTK)
2. **Tokenizar por sentenças** — com `sent_tokenize` (NLTK)
3. **Normalizar com `.lower()`** — para que "Python", "PYTHON" e "python" sejam iguais

> 💡 O `word_tokenize` do NLTK usa regras linguísticas (não é um simples `.split()`):
> separa pontuação, trata contrações e preserva padrões como "5-0" ou "$499.99".


In [ ]:
# =============================================
# SEÇÃO 1: TOKENIZAÇÃO POR PALAVRAS
# =============================================
# Objetivo: dividir texto cru em uma lista de tokens (palavras + pontuação).
# O word_tokenize usa regras gramaticais do idioma especificado.

from nltk.tokenize import word_tokenize, sent_tokenize

# Três textos de cenários distintos para observar comportamentos diferentes
texto_esportivo = "O Brasil ganhou 5-0! Que jogo incrível :)"
texto_preco     = "Comprei um celular. O preço era $499.99!"
texto_contato   = "E-mail: contato@empresa.com.br - Telefone: (11) 98765-4321"

print("=" * 70)
print("TOKENIZAÇÃO POR PALAVRAS (word_tokenize)")
print("=" * 70)

for rotulo, texto in [("Esportivo", texto_esportivo),
                       ("Preço",     texto_preco),
                       ("Contato",   texto_contato)]:
    # language='portuguese' ajusta as regras de tokenização para PT-BR
    tokens_palavras = word_tokenize(texto, language='portuguese')
    print(f"\n📝 Texto ({rotulo}): {texto}")
    print(f"   Tokens:  {tokens_palavras}")
    print(f"   Total:   {len(tokens_palavras)} tokens")

# =============================================
# SEÇÃO 2: TOKENIZAÇÃO POR SENTENÇAS
# =============================================
# Objetivo: dividir um parágrafo em frases completas.
# Útil para resumo automático ou análise de sentimento frase a frase.

texto_multiplas_frases = "O Brasil ganhou! A vitória foi inesperada. Que jogo emocionante."
sentencas = sent_tokenize(texto_multiplas_frases, language='portuguese')

print("\n" + "=" * 70)
print("TOKENIZAÇÃO POR SENTENÇAS (sent_tokenize)")
print("=" * 70)
print(f"\nTexto original: {texto_multiplas_frases}")
for indice, sentenca in enumerate(sentencas, 1):
    print(f"  Sentença {indice}: \"{sentenca}\"")
print(f"\n💡 Resultado: {len(sentencas)} sentenças detectadas pelo ponto final e exclamação.")

# =============================================
# SEÇÃO 3: NORMALIZAÇÃO COM .lower()
# =============================================
# Objetivo: converter tudo para minúsculas para que o modelo trate
# "Python", "PYTHON" e "python" como a MESMA palavra.
# Sem normalização, o vocabulário fica inflado com duplicatas.

texto_misturado = "Python é Linguagem. PYTHON É RÁPIDO. python é popular."
tokens_sem_normalizar = word_tokenize(texto_misturado, language='portuguese')
tokens_normalizados   = word_tokenize(texto_misturado.lower(), language='portuguese')

print("\n" + "=" * 70)
print("NORMALIZAÇÃO (.lower())")
print("=" * 70)
print(f"\nSem normalizar: {tokens_sem_normalizar}")
print(f"Com .lower():   {tokens_normalizados}")
print("\n💡 'Python', 'PYTHON' e 'python' agora são um token só: 'python'.")
print("   Isso reduz o vocabulário e melhora a qualidade da vetorização.")

print("\n👉 Próximo passo: Remover stopwords — palavras comuns que não agregam significado.")


---
# PARTE 4: Remoção de Stopwords

## 🚫 O que são Stopwords?

**Definição:** Stopwords são palavras extremamente comuns em um idioma que, sozinhas, geralmente **não carregam significado útil** para a análise.  
Exemplos em português: "o", "a", "de", "para", "em", "é", "e", "ou".

```
Frase original:       "O gato de estimação é muito bom"
Stopwords removidas:   O, de, é, muito
Resultado:             "gato estimação bom" ← só ficou o que importa!
```

### Por que remover stopwords no pipeline?

Porque na etapa seguinte (vetorização), cada palavra vira uma **coluna na matriz**.  
Se mantivermos stopwords, a matriz fica cheia de colunas inúteis que:
- 🎯 **Adicionam ruído** — dilui o sinal das palavras que realmente importam
- 📉 **Inflam a dimensionalidade** — mais colunas = mais memória e tempo
- ⚡ **Retardam o processamento** — o modelo gasta esforço com dados irrelevantes

### ⚠️ CUIDADO CRÍTICO: Stopwords × Análise de Sentimentos

Nem **sempre** remover stopwords é uma boa ideia! Palavras como **"não"**, **"mas"**, **"nem"**, **"nunca"** são classificadas como stopwords, mas carregam **significado crucial** em análise de sentimentos:

| Frase original | Sem stopwords | Sentimento real | Sentimento detectado |
|---------------|---------------|-----------------|---------------------|
| "**não** é bom" | "bom" | ❌ Negativo | ✅ Positivo (ERRO!) |
| "bom **mas** caro" | "bom caro" | ⚠️ Misto | ✅ Positivo (ERRO!) |
| "**nem** ruim **nem** bom" | "ruim bom" | ⚠️ Neutro | ❓ Confuso |
| "**nunca** mais compro" | "compro" | ❌ Negativo | ✅ Neutro/Positivo (ERRO!) |

> **Dica prática:** Em tarefas de sentimento, crie uma **lista customizada** que
> **preserva negações** ("não", "nem", "nunca") e **conectivos de contraste** ("mas", "porém").
> Vamos demonstrar essa técnica no código abaixo.


In [ ]:
# =============================================
# EXPLORANDO A LISTA DE STOPWORDS DO NLTK
# =============================================
# Objetivo: carregar as stopwords em português e verificar
# se palavras críticas para sentimento estão nessa lista.

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Usamos set() em vez de list() para buscas O(1) — muito mais rápido
stopwords_portugues = set(stopwords.words('portuguese'))

print("=" * 70)
print("LISTA DE STOPWORDS DO NLTK (PORTUGUÊS)")
print("=" * 70)
print(f"\nTotal de stopwords: {len(stopwords_portugues)}")
print(f"Primeiras 15 (ordem alfabética): {sorted(list(stopwords_portugues))[:15]}")

# Verificar palavras perigosas
print("\n⚠️ Verificação: palavras de NEGAÇÃO e CONTRASTE na lista de stopwords")
palavras_perigosas = ['não', 'mas', 'nem', 'nunca', 'nada', 'sem']
for palavra in palavras_perigosas:
    esta_na_lista = palavra in stopwords_portugues
    emoji = "🚨" if esta_na_lista else "✅"
    status = "SIM — será removida!" if esta_na_lista else "NÃO — está segura"
    print(f"   {emoji} '{palavra}': {status}")

print("\n💡 Conclusão: várias palavras essenciais para sentimento são stopwords!")
print("   Veja no próximo bloco o impacto prático disso.")


In [ ]:
# =============================================
# IMPACTO PRÁTICO: STOPWORDS EM REVIEWS DE SENTIMENTO
# =============================================
# Objetivo: mostrar como a remoção padrão de stopwords
# pode DESTRUIR o significado de reviews negativas.

lista_reviews = [
    "Adorei o produto! É de ótima qualidade.",            # Positiva pura
    "Não gostei. Produto de péssima qualidade.",          # ← "não" será removido!
    "É ok. Preço bom mas qualidade ruim.",                # ← "mas" será removido!
    "Nunca mais compro. Nem funciona direito.",           # ← "nunca" e "nem" removidos!
]

print("=" * 70)
print("EFEITO DA REMOÇÃO DE STOPWORDS EM REVIEWS")
print("=" * 70)

for indice, review in enumerate(lista_reviews, 1):
    # Passo 1: tokenizar e normalizar
    tokens_originais = word_tokenize(review.lower(), language='portuguese')

    # Passo 2: remover stopwords + pontuação (filtro padrão)
    # - token.isalpha() remove pontuação e números
    # - "not in stopwords_portugues" remove stopwords
    tokens_sem_stopwords = [
        token for token in tokens_originais
        if token not in stopwords_portugues and token.isalpha()
    ]

    print(f"\n📝 Review {indice}: \"{review}\"")
    print(f"   Tokens originais:  {tokens_originais}")
    print(f"   Sem stopwords:     {tokens_sem_stopwords}")

print("\n" + "=" * 70)
print("⚠️ PROBLEMAS DETECTADOS:")
print("   Review 2: 'não' removido → sobrou 'gostei péssima qualidade' (perde negação!)")
print("   Review 3: 'mas' removido → perde o contraste entre 'bom' e 'ruim'")
print("   Review 4: 'nunca' e 'nem' removidos → perde toda a negatividade!")
print("\n💡 Solução: usar uma lista CUSTOMIZADA de stopwords. Veja abaixo!")


In [ ]:
# =============================================
# SOLUÇÃO: LISTA CUSTOMIZADA DE STOPWORDS
# =============================================
# Objetivo: criar uma versão da lista que PRESERVA
# negações e conectivos de contraste.
# Técnica: operação de conjuntos (set difference).

# Palavras que queremos MANTER mesmo sendo stopwords oficiais
palavras_a_preservar = {'não', 'mas', 'nem', 'nunca', 'nada', 'sem', 'porém', 'contudo'}

# Criar lista customizada: stopwords MENOS as palavras a preservar
# Operação de conjuntos: A - B = tudo que está em A mas NÃO em B
stopwords_customizadas = stopwords_portugues - palavras_a_preservar

print("=" * 70)
print("LISTA CUSTOMIZADA DE STOPWORDS")
print("=" * 70)
print(f"\nStopwords originais:    {len(stopwords_portugues)} palavras")
print(f"Stopwords customizadas: {len(stopwords_customizadas)} palavras")
print(f"Palavras preservadas:   {sorted(palavras_a_preservar)}")

# Comparar o resultado nas reviews problemáticas
review_problematica = "Não gostei. Produto de péssima qualidade."
tokens = word_tokenize(review_problematica.lower(), language='portuguese')

tokens_filtro_padrao = [t for t in tokens if t not in stopwords_portugues and t.isalpha()]
tokens_filtro_custom = [t for t in tokens if t not in stopwords_customizadas and t.isalpha()]

print(f"\n📝 Review: \"{review_problematica}\"")
print(f"   Filtro padrão:     {tokens_filtro_padrao}")
print(f"   Filtro customizado: {tokens_filtro_custom}  ← 'não' preservado!")

print("\n✅ Com a lista customizada, mantemos as negações e o modelo")
print("   consegue distinguir 'gostei' de 'NÃO gostei'.")

print("\n👉 Próximo passo: Bag of Words — transformar o texto limpo em uma matriz numérica.")


---
# PARTE 5: Bag of Words (BoW)

## 👜 O que é Bag of Words?

**Definição:** Bag-of-Words é uma forma de representar texto como **números**, contando quantas vezes cada palavra aparece em cada documento. Ignora completamente a **ordem** das palavras.

### Por que importa no pipeline?

Até aqui temos tokens limpos (palavras sem stopwords). Mas modelos de Machine Learning **não entendem texto** — eles precisam de **números**. BoW é a forma mais simples e direta de fazer essa conversão.

### Analogia 🎒

Imagine jogar todas as palavras de uma frase dentro de uma mochila. Não importa a ordem — só importa **quantas de cada tipo** tem lá dentro:

```
Frase 1: "O gato subiu no telhado"  →  {gato: 1, subiu: 1, telhado: 1}
Frase 2: "No telhado o gato subiu"  →  {gato: 1, subiu: 1, telhado: 1}
⚠️ MESMA representação! BoW não se importa com ordem.
```

### Como funciona (3 passos)

```
PASSO 1 — Vocabulário: listar todas as palavras únicas do corpus
  Docs: "python é bom", "python é rápido"
  Vocab: {bom, é, python, rápido}

PASSO 2 — Contagem: para cada doc, contar frequência de cada palavra
  Doc1: bom(1) é(1) python(1) rápido(0) → [1, 1, 1, 0]
  Doc2: bom(0) é(1) python(1) rápido(1) → [0, 1, 1, 1]

PASSO 3 — Montar a Matriz BoW
         bom  é  python  rápido
  Doc1    1   1    1       0
  Doc2    0   1    1       1
```

### Propriedades

| ✅ Vantagens | ❌ Limitações |
|-------------|---------------|
| Simples e rápido | Perde ordem das palavras |
| Transparente (fácil de entender) | Matriz esparsa (muitos zeros) |
| Funciona bem para classificação | Não captura semântica |

> Na célula abaixo, usamos o `CountVectorizer` do scikit-learn, que faz os 3 passos automaticamente.


In [ ]:
# =============================================
# BAG OF WORDS COM CountVectorizer
# =============================================
# Objetivo: transformar reviews de texto em uma matriz numérica
# onde cada linha = review e cada coluna = palavra do vocabulário.
#
# O CountVectorizer faz tudo em uma chamada:
#   1. Aprende o vocabulário (todas as palavras únicas do corpus)
#   2. Conta a frequência de cada palavra em cada documento
#   3. Retorna uma matriz esparsa (eficiente em memória)

from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

print("=" * 70)
print("DEMONSTRAÇÃO: BAG OF WORDS (CountVectorizer)")
print("=" * 70)

# Corpus: cada string é um "documento" (review de produto)
reviews_produtos = [
    "Som excelente, muito bom, recomendo!",       # Positivo
    "Produto ruim, som horrível, não recomendo",  # Negativo
    "Bom custo-benefício, som bom",               # Positivo
    "Qualidade excelente, adorei",                # Positivo
    "Horrível, pior compra, muito ruim",          # Negativo
]
rotulos_sentimento = ["Positivo", "Negativo", "Positivo", "Positivo", "Negativo"]

print("\n📝 Reviews originais:")
for indice, (review, rotulo) in enumerate(zip(reviews_produtos, rotulos_sentimento), 1):
    emoji = "😊" if rotulo == "Positivo" else "😞"
    print(f"   {indice}. {emoji} [{rotulo}] {review}")

# Criar e ajustar o vetorizador BoW
# lowercase=True → converte para minúsculas antes de contar
vetorizador_bow = CountVectorizer(lowercase=True)

# fit_transform() = fit() + transform():
#   fit()       → aprende o vocabulário a partir dos textos
#   transform() → transforma cada texto em vetor usando esse vocabulário
matriz_bow = vetorizador_bow.fit_transform(reviews_produtos)

vocabulario_bow = vetorizador_bow.get_feature_names_out()  # lista de palavras do vocabulário

print(f"\n✨ Vocabulário criado automaticamente:")
print(f"   Total de palavras únicas: {len(vocabulario_bow)}")
print(f"   Palavras: {list(vocabulario_bow)}")

# Visualizar a matriz como DataFrame
# .toarray() converte a matriz esparsa em array denso (para exibir)
# Cada LINHA = uma review | Cada COLUNA = uma palavra | Cada CÉLULA = contagem
tabela_bow = pd.DataFrame(
    matriz_bow.toarray(),       # converter matriz esparsa → array numpy
    columns=vocabulario_bow,    # nomes das colunas = palavras
    index=[f"Review {i+1} ({r})" for i, r in enumerate(rotulos_sentimento)]
)

print("\n📊 Matriz Bag-of-Words:")
print(tabela_bow)

# Frequências totais: somar cada coluna (palavra) em todos os docs
frequencia_total = tabela_bow.sum(axis=0)       # axis=0 → soma vertical
top_5_palavras = frequencia_total.nlargest(5)    # 5 maiores

print("\n📈 Top 5 palavras mais frequentes no corpus:")
for palavra, freq in top_5_palavras.items():
    print(f"   '{palavra}': {int(freq)}x")

print("\n💡 Como interpretar esta matriz:")
print("   Linha  = documento (review)")
print("   Coluna = palavra do vocabulário")
print("   Valor 0 = palavra NÃO aparece naquele documento")
print("   Valor 1+ = quantas vezes a palavra aparece")
print("   Ex: 'bom' aparece em vários docs → frequente mas não distintiva")
print("       'adorei' aparece em 1 doc só → mais distintiva")

print("\n👉 Próximo passo: TF-IDF — pesos inteligentes que resolvem o problema das palavras comuns.")


---
# PARTE 6: TF-IDF (Term Frequency — Inverse Document Frequency)

## ⚖️ O que é TF-IDF?

**Definição:** TF-IDF é uma técnica que mede a **importância real de uma palavra** em um documento, levando em conta o quanto essa palavra é **comum ou rara** no corpus todo.

### Por que importa no pipeline? (Limitação do BoW)

No BoW, **todas as palavras têm o mesmo peso** — a contagem bruta trata "o" e "incrível" como igualmente relevantes. Mas isso é enganoso:

```
Doc 1: "O gato e o gato e o gato"   ← "o" aparece 3x → peso ALTO no BoW
Doc 2: "Machine Learning é incrível" ← "incrível" aparece 1x → peso BAIXO no BoW

❌ PROBLEMA: "o" é super COMUM (aparece em quase todo texto em português!)
✅ TF-IDF: "incrível" ganha peso MUITO maior que "o"
   porque "incrível" é RARO no corpus.
```

### Fórmula: TF-IDF = TF × IDF

| Componente | Nome | O que mede | Efeito no peso |
|-----------|------|-----------|----------------|
| **TF** | Term Frequency | Frequência da palavra **neste** documento | Alta frequência local → TF alto |
| **IDF** | Inverse Document Frequency | Raridade da palavra **no corpus todo** | Palavra rara → IDF alto |

```
Exemplo prático (corpus com 100 documentos):

  Palavra "Python":
    TF  = 3/7 = 0.43  (aparece 3x num doc de 7 palavras)
    IDF = log(100/20) = 1.61  (aparece em 20 dos 100 docs → relativamente rara)
    TF-IDF = 0.43 × 1.61 = 0.69  ← PESO ALTO — palavra importante!

  Palavra "é":
    TF  = 2/7 = 0.29
    IDF = log(100/95) = 0.05  (aparece em 95 dos 100 docs → muito comum)
    TF-IDF = 0.29 × 0.05 = 0.015  ← PESO BAIXO — palavra genérica.
```

> **Resumo em uma frase:** TF-IDF dá peso alto a palavras que são frequentes **neste** documento mas raras **no corpus geral**. Isso destaca automaticamente as palavras mais relevantes de cada texto.

### O que vamos fazer na célula abaixo:
Aplicar o `TfidfVectorizer` no mesmo estilo do `CountVectorizer` e ver como os pesos mudam.


In [ ]:
# =============================================
# TF-IDF COM TfidfVectorizer
# =============================================
# Objetivo: aplicar TF-IDF em avaliações de produtos e
# interpretar quais palavras são realmente importantes.
#
# O TfidfVectorizer funciona igual ao CountVectorizer:
#   fit_transform() → aprende vocabulário + calcula pesos TF-IDF
# A diferença é que os valores NÃO são contagens inteiras,
# mas decimais entre 0 e 1 que representam importância:
#   ~0.0 = palavra pouco relevante (muito comum no corpus)
#   ~1.0 = palavra muito relevante (rara e frequente no doc)

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

print("=" * 70)
print("DEMONSTRAÇÃO: TF-IDF (TfidfVectorizer)")
print("=" * 70)

corpus_avaliacoes = [
    'Celular excelente, câmera sensacional',   # Doc 1
    'Câmera péssima, celular ruim',            # Doc 2
    'Celular ok, preço bom',                   # Doc 3
]

print("\n📝 Corpus de avaliações:")
for indice, documento in enumerate(corpus_avaliacoes, 1):
    print(f"   {indice}. \"{documento}\"")

# Criar e ajustar o vetorizador TF-IDF
vetorizador_tfidf = TfidfVectorizer(lowercase=True)
matriz_tfidf = vetorizador_tfidf.fit_transform(corpus_avaliacoes)  # aprende + transforma
vocabulario_tfidf = vetorizador_tfidf.get_feature_names_out()

# Visualizar como tabela (arredondando para 4 casas decimais)
tabela_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),     # esparsa → densa para exibição
    columns=vocabulario_tfidf,
    index=[f"Doc {i+1}" for i in range(len(corpus_avaliacoes))]
)

print("\n📊 Matriz TF-IDF (cada valor = importância da palavra no documento):")
print(tabela_tfidf.round(4))

# Top 3 palavras mais importantes por documento
print("\n🏆 TOP 3 PALAVRAS MAIS RELEVANTES POR DOCUMENTO:")
print("-" * 70)
for indice_doc in range(len(corpus_avaliacoes)):
    pesos = matriz_tfidf[indice_doc].toarray()[0]     # vetor de pesos do doc
    # argsort() retorna índices em ordem crescente; [-3:] pega os 3 maiores;
    # [::-1] inverte para ordem decrescente
    indices_top3 = pesos.argsort()[-3:][::-1]
    print(f"\n📄 Doc {indice_doc + 1}: \"{corpus_avaliacoes[indice_doc]}\"")
    for posicao, idx_palavra in enumerate(indices_top3, 1):
        nome_palavra = vocabulario_tfidf[idx_palavra]
        peso_tfidf = pesos[idx_palavra]
        print(f"   {posicao}. '{nome_palavra}': {peso_tfidf:.4f}")

print("\n💡 Como interpretar:")
print("   • 'sensacional' tem peso ALTO no Doc 1 → só aparece lá (rara no corpus!)")
print("   • 'celular' tem peso BAIXO → aparece em TODOS os docs (comum demais!)")
print("   • TF-IDF destaca automaticamente o que torna cada documento ÚNICO.")

print("\n👉 Próximo passo: comparar BoW e TF-IDF lado a lado para visualizar a diferença.")


---
# PARTE 7: Comparação Prática — BoW vs TF-IDF

## 🔍 Qual a diferença na prática?

Nas partes 5 e 6 vimos BoW e TF-IDF separadamente. Agora vamos usar o **mesmo corpus** e colocar os resultados **lado a lado** para que a diferença fique evidente.

| Aspecto | BoW (`CountVectorizer`) | TF-IDF (`TfidfVectorizer`) |
|---------|------------------------|---------------------------|
| **O que calcula** | Contagem bruta de palavras | Importância relativa (TF × IDF) |
| **Tipo de valor** | Inteiros (0, 1, 2...) | Decimais (0.0 a ~1.0) |
| **Palavras comuns** | Peso alto (aparecem muito) | Peso **baixo** (são genéricas) |
| **Palavras raras** | Peso = 1 (só conta) | Peso **alto** (são únicas!) |
| **Melhor para** | Naive Bayes, tarefas simples | Busca, ranking, clustering, SVM |

> **Regra geral:** Comece com BoW por ser mais simples. Se precisar de resultados melhores, troque por TF-IDF. Ambos alimentam modelos de ML da mesma forma — a diferença está na **qualidade** da representação.


In [ ]:
# =============================================
# COMPARAÇÃO LADO A LADO: BoW vs TF-IDF
# =============================================
# Objetivo: usar o MESMO corpus com os dois vetorizadores
# e comparar os valores resultantes para ver a diferença real.

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

# Mesmo corpus para ambos
corpus_comparacao = [
    "Python é uma linguagem de programação",     # Doc 1
    "Python é rápido e poderoso",                 # Doc 2
    "Java é uma linguagem de programação",        # Doc 3
    "Machine Learning usa Python",                # Doc 4
]

print("=" * 70)
print("COMPARAÇÃO: BoW vs TF-IDF (mesmo corpus)")
print("=" * 70)

# --- BoW (contagem bruta) ---
vetorizador_bow_comp = CountVectorizer(lowercase=True)
matriz_bow_comp = vetorizador_bow_comp.fit_transform(corpus_comparacao)
tabela_bow_comp = pd.DataFrame(
    matriz_bow_comp.toarray(),
    columns=vetorizador_bow_comp.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus_comparacao))]
)

# --- TF-IDF (pesos inteligentes) ---
vetorizador_tfidf_comp = TfidfVectorizer(lowercase=True)
matriz_tfidf_comp = vetorizador_tfidf_comp.fit_transform(corpus_comparacao)
tabela_tfidf_comp = pd.DataFrame(
    matriz_tfidf_comp.toarray(),
    columns=vetorizador_tfidf_comp.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus_comparacao))]
)

print("\n📊 Matriz BoW (contagem bruta — valores inteiros):")
print(tabela_bow_comp)

print("\n📊 Matriz TF-IDF (importância ponderada — valores decimais):")
print(tabela_tfidf_comp.round(3))

# Análise das diferenças
print("\n" + "=" * 70)
print("🔎 ANÁLISE DAS DIFERENÇAS")
print("=" * 70)

print("\nPalavra 'é' (COMUM — aparece em 3 dos 4 docs):")
peso_medio_e = tabela_tfidf_comp['é'].mean()
print(f"   BoW    → contagem = 1 em cada doc (mesmo peso que qualquer outra)")
print(f"   TF-IDF → peso médio = {peso_medio_e:.3f} (BAIXO, porque é muito comum!)")

print("\nPalavra 'machine' (RARA — aparece em 1 doc apenas):")
peso_max_machine = tabela_tfidf_comp['machine'].max()
print(f"   BoW    → contagem = 1 (mesmo peso que 'é' — não diferencia!)")
print(f"   TF-IDF → peso = {peso_max_machine:.3f} (ALTO, porque é exclusiva do Doc 4!)")

print("\n✅ Conclusão:")
print("   BoW trata todas as palavras igualmente: 'é' = 'machine' = 1.")
print("   TF-IDF reduz o peso de palavras comuns e destaca as raras.")
print("   Na prática, isso melhora a qualidade da entrada para modelos de ML.")

print("\n👉 Próximo passo: usar essas representações em um modelo real de classificação!")


---
# PARTE 8: Classificação de Sentimentos

## 🤖 Fechando o Pipeline: Texto → Números → Modelo → Predição

Até aqui, transformamos texto em números (BoW e TF-IDF). Agora vamos usar esses números como **entrada para modelos de Machine Learning** que classificam reviews como positivas ou negativas.

### Modelos que vamos usar

| Modelo | O que faz | Por que é bom para texto |
|--------|----------|-------------------------|
| **Naive Bayes** (`MultinomialNB`) | Calcula a probabilidade de cada classe usando o Teorema de Bayes | Rápido, simples, funciona surpreendentemente bem com BoW/TF-IDF |
| **Logistic Regression** (`LogisticRegression`) | Encontra uma fronteira linear que separa as classes | Mais robusto com mais dados, excelente baseline |

### Fluxo desta célula:
```
Textos de treino (10 reviews rotuladas)
    ↓ TfidfVectorizer.fit_transform()     ← aprende vocabulário + vetoriza
Matriz TF-IDF de treino
    ↓ modelo.fit()                         ← treina os classificadores
Modelos treinados
    ↓ TfidfVectorizer.transform()          ← vetoriza novas reviews (SEM reaprender)
    ↓ modelo.predict()                     ← classifica
Predições: Positivo ou Negativo
```

> **⚠️ Detalhe importante:** Na hora de vetorizar os dados de teste, usamos `.transform()` (e **não** `.fit_transform()`). Se usássemos `fit_transform`, o vetorizador criaria um **novo vocabulário** baseado nos dados de teste, o que quebraria a compatibilidade com o modelo treinado.

> **Nota:** Nosso dataset tem apenas 10 exemplos de treino — é didático. Em projetos reais, use milhares de exemplos e `train_test_split` para separar treino e teste.


In [ ]:
# =============================================
# CLASSIFICAÇÃO DE SENTIMENTOS: PIPELINE COMPLETO
# =============================================
# Objetivo: treinar dois modelos (Naive Bayes e Logistic Regression)
# em reviews rotuladas e testar em reviews novas.
# Pipeline: texto → TF-IDF → modelo → predição.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
import numpy as np

print("=" * 70)
print("CLASSIFICAÇÃO DE SENTIMENTOS")
print("=" * 70)

# ---- PASSO 1: DADOS DE TREINO ----
# 10 reviews rotuladas manualmente: 5 positivas (1) e 5 negativas (0)
textos_treino = [
    "Produto excelente, adorei, super recomendo",          # Positivo
    "Muito bom, qualidade incrível, comprem",              # Positivo
    "Ótimo produto, entrega rápida, voltarei a comprar",   # Positivo
    "Gostei muito, superou expectativas",                  # Positivo
    "Produto maravilhoso, melhor compra que fiz",          # Positivo
    "Péssimo produto, não funciona, devolvi",              # Negativo
    "Horrível, quebrou no segundo dia",                    # Negativo
    "Muito ruim, qualidade péssima, não comprem",          # Negativo
    "Produto terrível, parece falsificado",                # Negativo
    "Detestei, pior compra, dinheiro jogado fora",         # Negativo
]
rotulos_treino = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]  # 1=Positivo, 0=Negativo

print("\n📝 Dados de treino:")
for texto, rotulo in zip(textos_treino, rotulos_treino):
    emoji = "😊" if rotulo == 1 else "😞"
    sentimento = "Positivo" if rotulo == 1 else "Negativo"
    print(f"   {emoji} [{sentimento}] {texto}")

# ---- PASSO 2: VETORIZAÇÃO TF-IDF ----
# fit_transform nos TREINO: aprende o vocabulário E cria os vetores
vetorizador_classificacao = TfidfVectorizer(lowercase=True)
matriz_treino = vetorizador_classificacao.fit_transform(textos_treino)

print(f"\n📊 Matriz TF-IDF de treino:")
print(f"   Formato: {matriz_treino.shape[0]} documentos × {matriz_treino.shape[1]} palavras")

# ---- PASSO 3: TREINAR MODELOS ----
# Naive Bayes: bom para dados esparsos (matrizes com muitos zeros)
modelo_naive_bayes = MultinomialNB()
modelo_naive_bayes.fit(matriz_treino, rotulos_treino)  # fit = treinar

# Logistic Regression: mais robusto, encontra fronteira linear entre classes
modelo_logistic = LogisticRegression(max_iter=1000)
modelo_logistic.fit(matriz_treino, rotulos_treino)

print("✅ Modelos treinados com sucesso!")

# ---- PASSO 4: PREDIÇÃO EM NOVAS REVIEWS ----
reviews_teste = [
    "Amei o produto, excelente qualidade",    # Esperado: Positivo
    "Não gostei, muito ruim",                 # Esperado: Negativo
    "Bom preço e boa qualidade",              # Esperado: Positivo
    "Péssimo, nunca mais compro",             # Esperado: Negativo
    "Produto ok, nada demais",                # Esperado: Neutro/ambíguo
]

# ATENÇÃO: usamos transform() (NÃO fit_transform!)
# O vocabulário já foi aprendido no fit_transform acima.
# Se fizermos fit_transform aqui, o vocabulário muda e quebra tudo.
matriz_teste = vetorizador_classificacao.transform(reviews_teste)

# Gerar predições com ambos os modelos
predicoes_nb = modelo_naive_bayes.predict(matriz_teste)
predicoes_lr = modelo_logistic.predict(matriz_teste)

print("\n" + "=" * 70)
print("PREDIÇÕES PARA NOVAS REVIEWS")
print("=" * 70)

for review, pred_nb, pred_lr in zip(reviews_teste, predicoes_nb, predicoes_lr):
    sentimento_nb = "Positivo 😊" if pred_nb == 1 else "Negativo 😞"
    sentimento_lr = "Positivo 😊" if pred_lr == 1 else "Negativo 😞"
    print(f"\n📝 \"{review}\"")
    print(f"   → Naive Bayes:         {sentimento_nb}")
    print(f"   → Logistic Regression: {sentimento_lr}")

print("\n" + "=" * 70)
print("💡 OBSERVAÇÕES IMPORTANTES:")
print("   • Com apenas 10 exemplos de treino, os modelos já classificam razoavelmente!")
print("   • Naive Bayes: simples, rápido, ótimo ponto de partida para texto.")
print("   • Logistic Regression: geralmente mais preciso com mais dados.")
print("   • Em projetos reais: use milhares de exemplos + train_test_split().")
print("   • Dica: experimente trocar TfidfVectorizer por CountVectorizer acima")
print("     e compare os resultados!")

print("\n👉 Próximo passo: Word Embeddings — uma forma mais sofisticada de representar texto.")


---
# PARTE 9: Word Embeddings (Conceitual + Demo)

## 🧠 O que são Word Embeddings?

**Definição:** Word Embeddings são uma forma de representar palavras como **vetores densos de números reais** (geralmente 50 a 300 dimensões), onde palavras com significados parecidos ficam **próximas** no espaço vetorial.

### Por que importa no pipeline?

BoW e TF-IDF são ótimos, mas têm uma limitação fundamental: **não entendem semântica**. Para eles, "bom" e "ótimo" são palavras completamente diferentes (colunas separadas na matriz). Word Embeddings resolvem isso:

```
BoW/TF-IDF: Vetor ESPARSO (muitos zeros, sem semântica)
  Exemplo: [1, 0, 0, 1, 0, 0, 0, ..., 0, 0]   dimensão ~10.000+
  → "bom" e "ótimo" são colunas diferentes, sem relação

Embeddings: Vetor DENSO (valores reais, com semântica)
  Exemplo: [0.25, -0.8, 0.1, ..., 0.9, -0.3]   dimensão 50-300
  → "bom" e "ótimo" têm vetores PRÓXIMOS (significados similares!)
```

### Propriedades Mágicas ✨

```
"gato" ≈ "felino"         → vetores próximos no espaço!
"rei" - "homem" + "mulher" ≈ "rainha"  → aritmética semântica funciona!
```

### Técnicas de Embedding

| Técnica | Origem | Destaque |
|---------|--------|----------|
| **Word2Vec** | Google (2013) | Aprende pelo contexto (Skip-gram / CBOW), rápido |
| **GloVe** | Stanford | Combina frequências globais com contexto local |
| **FastText** | Facebook/Meta | Usa subpalavras (melhor para idiomas com muita morfologia) |
| **BERT/GPT** | Google/OpenAI | Transformers — estado-da-arte atual, contexto dinâmico |

### Quando usar cada representação?

| Situação | Recomendação |
|----------|-------------|
| Classificação simples, poucos dados, interpretabilidade | **BoW ou TF-IDF** |
| Precisa de semântica, dados medianos | **Word2Vec / FastText** |
| Estado-da-arte, tarefas complexas, muitos dados | **BERT / GPT** |

> **Para iniciantes:** BoW e TF-IDF resolvem a grande maioria dos problemas do dia a dia. Embeddings são o próximo nível quando você precisar capturar significado e relações entre palavras.

### O que vamos fazer abaixo:
Treinar um modelo Word2Vec simples com Gensim e explorar similaridades entre palavras.


In [ ]:
# =============================================
# WORD EMBEDDINGS COM WORD2VEC (GENSIM)
# =============================================
# Objetivo: treinar um modelo Word2Vec pequeno, visualizar
# os vetores gerados e explorar similaridades entre palavras.
#
# Como o Word2Vec aprende:
#   Ele analisa o CONTEXTO (palavras vizinhas) de cada palavra.
#   Palavras que aparecem em contextos parecidos → vetores parecidos.
#   Ex: "gato" e "cachorro" aparecem em frases como "o ___ corre"
#       → seus vetores ficam próximos no espaço!

import numpy as np

try:
    from gensim.models import Word2Vec

    print("=" * 70)
    print("WORD EMBEDDINGS COM WORD2VEC")
    print("=" * 70)

    # Dados de treino: sentenças já tokenizadas (lista de listas de palavras)
    # Em projetos reais, você usaria um corpus MUITO maior (milhares de frases)
    sentencas_treino = [
        ['o', 'gato', 'preto', 'dorme'],
        ['o', 'gato', 'branco', 'corre'],
        ['o', 'cachorro', 'preto', 'corre'],
        ['gato', 'e', 'cachorro', 'são', 'animais'],
        ['python', 'é', 'linguagem', 'de', 'programação'],
        ['java', 'é', 'linguagem', 'de', 'programação'],
        ['python', 'é', 'rápido', 'e', 'poderoso'],
    ]

    # Treinar o modelo
    modelo_w2v = Word2Vec(
        sentences=sentencas_treino,
        vector_size=50,  # cada palavra → vetor de 50 dimensões
        window=3,        # contexto: considera 3 palavras antes e depois
        min_count=1,     # inclui todas as palavras (dataset pequeno)
        workers=1,       # 1 thread para reprodutibilidade
    )

    print(f"\n📊 Informações do Modelo:")
    print(f"   Vocabulário: {len(modelo_w2v.wv)} palavras aprendidas")
    print(f"   Dimensão dos vetores: {modelo_w2v.vector_size}")

    # Visualizar o vetor de uma palavra
    palavra_exemplo = "gato"
    vetor_exemplo = modelo_w2v.wv[palavra_exemplo]  # acessar o vetor da palavra
    print(f"\n🧠 Exemplo de Embedding para '{palavra_exemplo}':")
    print(f"   Primeiras 10 dimensões: {np.round(vetor_exemplo[:10], 4)}")
    print(f"   Magnitude do vetor:     {np.linalg.norm(vetor_exemplo):.4f}")
    print(f"   (cada palavra é representada por {modelo_w2v.vector_size} números como esses)")

    # Explorar similaridades
    print(f"\n🔗 Palavras mais SIMILARES a cada termo:")
    print("-" * 55)
    for palavra_consulta in ['gato', 'python', 'programação']:
        if palavra_consulta in modelo_w2v.wv:
            # most_similar retorna as N palavras com vetores mais próximos
            resultados = modelo_w2v.wv.most_similar(palavra_consulta, topn=3)
            print(f"\n   '{palavra_consulta}':")
            for palavra_similar, score in resultados:
                # score = cosseno entre os vetores (1.0 = idênticos, 0.0 = sem relação)
                print(f"      → {palavra_similar:<15} (similaridade: {score:.4f})")

    print("\n💡 Interpretação:")
    print("   • 'gato' e 'cachorro' → similares (ambos aparecem em 'o ___ corre')")
    print("   • 'python' e 'java' → similares (ambos em '___ é linguagem de programação')")
    print("   • Com um corpus maior, as similaridades seriam MUITO mais precisas.")
    print("   • Em produção, usamos embeddings pré-treinados (GloVe, FastText)")
    print("     ou modelos como BERT que já foram treinados em bilhões de textos.")

except ImportError:
    print("⚠️ Gensim não está instalado. Para usar Word2Vec:")
    print("   !pip install gensim")
    print("\n   Mas não se preocupe: tudo que fizemos com BoW e TF-IDF")
    print("   nas partes anteriores é suficiente para a maioria dos projetos!")


---
# PARTE 10: Conclusão

## 🎯 Resumo do Pipeline PLN

Ao longo deste notebook, percorremos o **pipeline completo** de PLN, desde o texto bruto até a classificação:

```
1. TOKENIZAÇÃO      → Dividir texto em palavras/sentenças (word_tokenize)
2. NORMALIZAÇÃO     → Converter para minúsculas (.lower())
3. STOPWORDS        → Remover palavras genéricas (com cuidado em sentimentos!)
4. VETORIZAÇÃO
   ├── BoW          → Contagem simples de frequências (CountVectorizer)
   ├── TF-IDF       → Pesos inteligentes por raridade (TfidfVectorizer)
   └── Embeddings   → Vetores semânticos densos (Word2Vec)
5. CLASSIFICAÇÃO    → Modelos de ML aplicados (MultinomialNB, LogisticRegression)
```

## 📋 Checklist do que cobrimos

| Tópico | Conceito | Ferramenta Python | Exemplo prático |
|--------|---------|-------------------|----------------|
| Tokenização por palavras | ✅ | `word_tokenize` | ✅ 3 cenários |
| Tokenização por sentenças | ✅ | `sent_tokenize` | ✅ |
| Normalização | ✅ | `.lower()` | ✅ Antes/depois |
| Stopwords (padrão) | ✅ | `nltk.corpus.stopwords` | ✅ 4 reviews |
| Stopwords (customizadas) | ✅ | Operações de `set` | ✅ Preservar negações |
| Bag of Words | ✅ | `CountVectorizer` | ✅ Matriz + top palavras |
| TF-IDF | ✅ | `TfidfVectorizer` | ✅ Matriz + top por doc |
| BoW vs TF-IDF | ✅ | Comparação lado a lado | ✅ Mesmo corpus |
| Classificação (Naive Bayes) | ✅ | `MultinomialNB` | ✅ Treino + predição |
| Classificação (Log. Reg.) | ✅ | `LogisticRegression` | ✅ Treino + predição |
| Word Embeddings | ✅ | `Word2Vec` (Gensim) | ✅ Similaridade |

## 🚀 Próximos Passos

Para continuar aprendendo, veja o notebook **IA_Machine_Learning.ipynb** que contém:
- 🤖 Fundamentos de Inteligência Artificial
- 📈 Modelos de Machine Learning em profundidade
- 🧠 Fundamentos Matemáticos
- ⚡ Limitações e Ética da IA

## 📖 Recursos Recomendados

| Tipo | Recurso |
|------|---------|
| 📚 Livro | *Speech and Language Processing* (Jurafsky & Martin) |
| 🎓 Curso | Natural Language Processing — Coursera / DeepLearning.AI |
| 🐍 Docs | [NLTK](https://www.nltk.org/) · [scikit-learn](https://scikit-learn.org/) |
| 🤗 Comunidade | [Hugging Face](https://huggingface.co/) — transformers modernos |

---

> **Parabéns!** 🎉 Você percorreu o pipeline completo de PLN, desde texto bruto até classificação de sentimentos. Agora tem uma base sólida para explorar projetos mais complexos.
